# pj-Flim movie agent — Colab agent v0.4

Qwen2.5:7b/Ollama + LangChain + multilingual E5/FAISS. Chạy từ trên xuống trong notebook. Đây là agent tìm phim và dẫn nguồn có giới hạn, website đã có tra cứu catalog/FAQ; notebook này thử Qwen độc lập, chưa huấn luyện model.

Notebook độc lập, không cần push GitHub. GPU/model trên Colab chưa được người tạo notebook chạy nghiệm thu thay bạn. Chọn dữ liệu/nguồn ở cell 1 trước khi Run all.

Nguồn: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Ollama Linux](https://docs.ollama.com/linux), [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama), [E5](https://huggingface.co/intfloat/multilingual-e5-small).


## 1. Chọn dữ liệu và nguồn

Mặc định dùng 4 phim giả do dự án tạo. Đổi USE_DEMO=False để nạp snapshot public của pj-Flim đã được phép xử lý cloud/embedding. Không đưa credential, lịch sử xem hoặc signed URL vào notebook. Source đi kèm có checksum; không cần GitHub đã cập nhật.


In [ ]:
# 1. Cấu hình + bộ source đi kèm: không cần clone/push GitHub.
USE_DEMO = True
REQUIRE_GPU = True
SHOW_CHAT_PANEL = True  # Ô nhập và nút hỏi ở cell 8; không cài thêm package vào kernel.
CONTEXT_TOKENS = 8192  # Nếu OOM: thử 4096, đo lại smoke và chất lượng.
ENABLE_OMDB = True  # Đọc OMDB_API_KEY từ Colab Secrets khi dùng catalog thật.
OMDB_MAPPINGS = {}  # UUID: {"imdb_id":"tt1375666","verified":True}; chỉ sau đối chiếu đúng tên/năm/loại.
ENABLE_TMDB = False
ENABLE_WEB_DISCOVERY = False  # Chỉ tìm URL; không đọc/copy bài review.
TMDB_MAPPINGS = {}  # UUID: {"kind":"movie" hoặc "tv","id":123,"verified":True}; chỉ sau khi đối chiếu đúng phim.

import base64, hashlib, json, platform, sys
from pathlib import Path
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("Mở notebook trong Google Colab Linux; notebook không cài Ollama trên PC.")
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11+ required")
WORKSPACE = Path("/content/pj-flim-agent")
SOURCE = WORKSPACE / "source"
SOURCE.mkdir(parents=True, exist_ok=True)
encoded = "..."
payload = base64.b64decode(encoded)
if hashlib.sha256(payload).hexdigest() != "0168945806ffcce5fbfa4cbbba5564c07807c520925bd80a5cabe236f34f703f":
    raise RuntimeError("BUNDLE_HASH_INVALID")
sources = json.loads(payload)
for relative_name, content in sources.items():
    target = (SOURCE / relative_name).resolve()
    if not target.is_relative_to(SOURCE.resolve()):
        raise RuntimeError("BUNDLE_PATH_INVALID")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
MODEL_CONFIG = json.loads((SOURCE / "model-versions.json").read_text())
print("Source bundle verified. Demo:", USE_DEMO, "OMDb:", ENABLE_OMDB, "TMDB:", ENABLE_TMDB, "Web:", ENABLE_WEB_DISCOVERY)


## 2. Kiểm runtime

Chọn Runtime → Change runtime type → GPU rồi chạy. Tài nguyên Colab không bảo đảm. Bản 7B cần đo VRAM/context thực, không coi file 4,7 GB là toàn bộ bộ nhớ.


In [ ]:
# Stdlib only: dependencies are installed in the next cell.
import platform, shutil, subprocess
HARDWARE = {"python": platform.python_version(), "system": platform.system(),
            "free_disk_gb": round(shutil.disk_usage("/content").free / 10**9, 2), "gpus": []}
try:
    report = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=10, check=True)
    HARDWARE["gpus"] = report.stdout.strip().splitlines()
except (OSError, subprocess.SubprocessError):
    pass
display(HARDWARE)
if REQUIRE_GPU and not HARDWARE["gpus"]:
    raise RuntimeError("GPU_NOT_ASSIGNED: chọn GPU trong Runtime và chạy lại. CPU thử nghiệm: REQUIRE_GPU=False.")
if HARDWARE["free_disk_gb"] < 10:
    raise RuntimeError("DISK_TOO_SMALL: cần khoảng trống cho Ollama/model/dependencies.")


## 3. Cài dependency

Cài trong venv riêng, không thay thư viện của kernel Colab. Agent chạy trong subprocess giữ trạng thái qua pipe, không mở cổng. E5 dùng Torch CPU; Qwen dùng GPU qua Ollama. Không cần restart vì NumPy của kernel khác phiên bản. Không có telemetry/tracing ứng dụng chủ động bật; tắt LangSmith tracing.


In [ ]:
# Only stdlib code is imported into the Colab kernel; all ML packages live in a venv.
from pjflim_agent.colab_bridge import prepare_environment, NotebookSession
if globals().get("CHAT_PANEL") is not None:
    CHAT_PANEL.close()
    CHAT_PANEL = None
if "session" in globals():
    session.close()
print("Đang chuẩn bị môi trường riêng. Lần đầu có thể mất vài phút; không thay NumPy của Colab.")
environment = prepare_environment(WORKSPACE, SOURCE)
versions = environment["versions"]
display(environment)

def show_agent_output(event):
    if event["kind"] == "text":
        print(event["value"], end="")
    else:
        display(event["value"])

session = NotebookSession(environment["python"], SOURCE, WORKSPACE, output=show_agent_output)
session.execute("pass", {
    "USE_DEMO": USE_DEMO, "REQUIRE_GPU": REQUIRE_GPU, "CONTEXT_TOKENS": CONTEXT_TOKENS,
    "ENABLE_OMDB": ENABLE_OMDB, "ENABLE_TMDB": ENABLE_TMDB, "ENABLE_WEB_DISCOVERY": ENABLE_WEB_DISCOVERY,
    "OMDB_MAPPINGS": OMDB_MAPPINGS, "TMDB_MAPPINGS": TMDB_MAPPINGS,
    "MODEL_CONFIG": MODEL_CONFIG, "HARDWARE": HARDWARE, "versions": versions,
})
print("Môi trường riêng đã sẵn sàng. Tiếp tục cell 4; không cần restart vì NumPy.")


## 4. Ollama + Qwen

Cell này cài Ollama phiên bản đã chọn và tải Qwen2.5:7b nhiều GB vào runtime Google. Chỉ bind 127.0.0.1:11434, không tunnel hoặc public model. Chỉ tái sử dụng/dừng process notebook sở hữu. Runtime được reset có thể cần tải lại.


In [ ]:
session.execute("import shutil\nfrom pjflim_agent.catalog import AgentError\nfrom pjflim_agent.runtime import start_ollama, pull_model\n# Chỉ cài trong Colab. Kiểm/cài curl + zstd trước installer; không cần restart NumPy ở đây.\nprint(\"Đang kiểm tra/cài Ollama trong runtime Colab; có thể mất vài phút.\")\ntry:\n    display(start_ollama(WORKSPACE))\nexcept AgentError as error:\n    hints = {\n        \"OLLAMA_INSTALL_TOOLS_MISSING\": \"Thiếu curl/zstd và không tự cài được bằng apt-get quyền root.\",\n        \"OLLAMA_DEPENDENCY_TIMEOUT\": \"apt-get quá thời gian; kiểm network runtime rồi chạy lại cell.\",\n        \"OLLAMA_DEPENDENCY_INSTALL_FAILED\": \"apt-get lỗi; log riêng tại install-dependencies.log.\",\n        \"OLLAMA_INSTALL_DOWNLOAD_FAILED\": \"Không tải được script HTTPS từ ollama.com; kiểm network/SSL.\",\n        \"OLLAMA_INSTALL_DOWNLOAD_TIMEOUT\": \"Tải script quá thời gian; kiểm network rồi chạy lại cell.\",\n        \"OLLAMA_INSTALL_DOWNLOAD_INVALID\": \"Script trả về rỗng hoặc quá giới hạn kích thước.\",\n        \"OLLAMA_INSTALL_SCRIPT_FAILED\": \"Installer thoát lỗi; log riêng tại install.log.\",\n        \"OLLAMA_INSTALL_TIMEOUT\": \"Installer vượt 15 phút; kiểm network/dung lượng runtime.\",\n        \"OLLAMA_NOT_INSTALLED\": \"Installer kết thúc nhưng chưa tìm thấy executable ollama.\",\n    }\n    display({\"code\": error.code, \"hint\": hints.get(error.code, \"Xem hướng dẫn lỗi trong README agent.\"),\n             \"missing_install_tools\": [t for t in (\"curl\", \"zstd\") if not shutil.which(t)]})\n    raise RuntimeError(error.code) from None\nprint(\"Ollama đã khởi động; đang tải/kiểm model Qwen (nhiều GB).\")\nMODEL_MANIFEST = pull_model(WORKSPACE)\ndisplay(MODEL_MANIFEST)\n")


## 5. Smoke model thật

Kiểm tiếng Việt, tool calling, JSON schema và digest/GPU offload của model sau inference bằng /api/ps; nếu lỗi hãy dừng tại đây. Nhãn kiểm thử trong tài liệu repo chưa thay bằng chứng Colab. Không lưu chain-of-thought.


In [ ]:
session.execute("import time\nfrom pydantic import BaseModel, ConfigDict\nfrom langchain_core.tools import StructuredTool\nfrom pjflim_agent.runtime import make_model, model_runtime\n# A failed rerun must not leave the previous success flag behind.\nglobals().pop(\"MODEL_SMOKE\", None)\nglobals().pop(\"MODEL_RUNTIME\", None)\nllm = make_model(context_tokens=CONTEXT_TOKENS)\nstarted = time.monotonic()\nhello = llm.invoke(\"Trả lời tiếng Việt một câu ngắn: Bạn là trợ lý tìm phim.\")\nif not isinstance(hello.content, str) or not hello.content.strip():\n    raise RuntimeError(\"MODEL_SMOKE_EMPTY\")\nprint(\"Smoke tiếng Việt:\", hello.content[:200])\n\nclass SmokeArgs(BaseModel):\n    model_config = ConfigDict(extra=\"forbid\", strict=True)\n    title_id: str\n\nsmoke_tool = StructuredTool.from_function(\n    func=lambda title_id: {\"title_id\": title_id},\n    name=\"lookup_demo_title\", description=\"Tra cứu phim demo bằng title_id.\", args_schema=SmokeArgs)\nprobe = llm.bind_tools([smoke_tool]).invoke(\n    \"Hãy gọi lookup_demo_title với title_id chính xác là demo-sao-hoa. Không trả văn bản.\")\nif len(probe.tool_calls) != 1 or probe.tool_calls[0][\"name\"] != \"lookup_demo_title\" or probe.tool_calls[0][\"args\"] != {\"title_id\": \"demo-sao-hoa\"}:\n    raise RuntimeError(\"MODEL_TOOL_SMOKE_FAILED: chưa thể coi model đã gọi tool đúng.\")\n\nclass SmokeJson(BaseModel):\n    ok: bool\n\ntyped = llm.with_structured_output(SmokeJson, method=\"json_schema\").invoke(\"Trả JSON với ok=true.\")\nif not isinstance(typed, SmokeJson) or typed.ok is not True:\n    raise RuntimeError(\"MODEL_JSON_SMOKE_FAILED\")\nMODEL_RUNTIME = model_runtime(MODEL_MANIFEST[\"digest\"], require_gpu=REQUIRE_GPU)\nMODEL_SMOKE = {\"nonempty_text\": True, \"language_needs_human_review\": True, \"tool_call\": True, \"json_schema\": True,\n               \"duration_seconds\": round(time.monotonic()-started, 2), \"runtime\": MODEL_RUNTIME}\ndisplay(MODEL_SMOKE)\n")


## 6. Nạp catalog

Demo mặc định không phải catalog thật. Với dữ liệu pj-Flim, chạy npm.cmd run agent:export-catalog trên PC khi API local đang chạy rồi chọn file .local/agent/catalog.snapshot.json. Snapshot hạn 24 giờ, có hash/schema và giới hạn 1.000 title. Đây là dữ liệu đủ điều kiện ở thời điểm xuất, không kiểm live.


In [ ]:
INPUT_NAME = ""
if not USE_DEMO:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("SELECT_ONE_SNAPSHOT_OR_CHECKPOINT")
    INPUT_NAME, input_content = next(iter(uploaded.items()))
    if len(input_content) > 8_000_000:
        raise RuntimeError("UPLOAD_TOO_LARGE")
    (WORKSPACE / "input-upload").write_bytes(input_content)
    del input_content, uploaded
session.execute("from pjflim_agent.catalog import Catalog, create_demo\nfrom pjflim_agent.checkpoint import restore_checkpoint\nCHECKPOINT_RESTORED = False\nif USE_DEMO:\n    catalog = create_demo(WORKSPACE / \"catalog.snapshot.json\")\nelse:\n    # Parent kernel uploads only one bounded JSON/ZIP to this fixed path.\n    name = INPUT_NAME\n    content = (WORKSPACE / \"input-upload\").read_bytes()\n    if name.lower().endswith(\".zip\"):\n        if len(content) > 8_000_000:\n            raise RuntimeError(\"CHECKPOINT_TOO_LARGE\")\n        archive_path = WORKSPACE / \"input-checkpoint.zip\"\n        archive_path.write_bytes(content)\n        restore_checkpoint(archive_path, WORKSPACE, expected_model_digest=MODEL_MANIFEST[\"digest\"])\n        CHECKPOINT_RESTORED = True\n    elif name.lower().endswith(\".json\"):\n        if len(content) > 5_000_000:\n            raise RuntimeError(\"SNAPSHOT_TOO_LARGE\")\n        (WORKSPACE / \"catalog.snapshot.json\").write_bytes(content)\n    else:\n        raise RuntimeError(\"JSON_OR_ZIP_REQUIRED\")\n    catalog = Catalog.load(WORKSPACE / \"catalog.snapshot.json\")\n    del content\n\ndisplay({\"titles\": len(catalog.by_id), \"demo\": catalog.snapshot.demo,\n         \"snapshot_at\": catalog.snapshot.generated_at, \"expires_at\": catalog.snapshot.expires_at,\n         \"truncated\": catalog.snapshot.truncated, \"live_eligibility_checked\": False})\nif catalog.snapshot.demo:\n    print(\"Đây là phim giả để thử agent; không phải phim đang có trên website.\")\n", {"INPUT_NAME": INPUT_NAME, "USE_DEMO": USE_DEMO})


## 7. E5 + FAISS CPU

Pin embedding revision, dùng query:/passage:, normalize và exact inner product trên candidate đã lọc. Hợp keyword + semantic bằng RRF. Checkpoint lưu vectors.npy allow_pickle=False, kiểm hash/shape/version rồi dựng index; không load pickle hoặc FAISS binary từ nguồn tùy ý.


In [ ]:
session.execute("from pjflim_agent.retrieval import HybridSearch\n# E5 chạy CPU. Qwen dùng GPU nếu runtime/offload phù hợp.\nsearch = HybridSearch(catalog, embedding_revision=MODEL_CONFIG[\"embeddingRevision\"],\n                      checkpoint_folder=WORKSPACE if CHECKPOINT_RESTORED else None)\nsearch.save(WORKSPACE)\nprint(\"E5/FAISS sẵn sàng:\", search.vectors.shape)\ndisplay([{\"id\": t.id, \"name\": t.name} for t in search.search(\"khoa học viễn tưởng\", limit=3)])\n")


## 8. Hỏi agent

Dùng ô nhập/nút Hỏi agent ở cuối cell8 hoặc đổi QUESTION rồi chạy lại cell8 nếu runtime không có ipywidgets. Mỗi câu hỏi độc lập; không ghi hội thoại vào checkpoint. Kết quả dễ đọc bằng tiếng Việt; mỗi câu hỏi là một lượt độc lập. CATALOG không gọi nguồn ngoài. HYBRID ưu tiên OMDb: thêm OMDB_API_KEY vào Colab Secrets và xác minh OMDB_MAPPINGS để lấy IMDb rating/Metascore. Tìm candidate bằng search_omdb_metadata; không tự duyệt mapping. TMDB vẫn tùy chọn với ENABLE_TMDB=True/TMDB_API_READ_TOKEN. ENABLE_WEB_DISCOVERY chỉ tìm URL theo tên phim; không đọc snippet/toàn văn. Có adapter OMDb lookup theo mapping; chưa bulk importer, Metacritic User Score, RT hoặc review summary. Demo chặn lookup ngoài.


In [ ]:
# Nhập câu hỏi bên dưới hoặc dùng ô chat; mỗi câu hỏi độc lập.
QUESTION = "Tìm phim khoa học viễn tưởng nhẹ nhàng, tránh spoiler."
MODE = "CATALOG" if USE_DEMO else "HYBRID"
def ask_notebook(question, mode="CATALOG", quiet=True):
    secrets = {}
    if not USE_DEMO and (ENABLE_OMDB or ENABLE_TMDB):
        from google.colab import userdata
        for enabled, key in [(ENABLE_OMDB, "OMDB_API_KEY"), (ENABLE_TMDB, "TMDB_API_READ_TOKEN")]:
            if enabled:
                try:
                    secrets[key] = userdata.get(key) or ""
                except Exception:
                    if not quiet: print(key + "_NOT_CONFIGURED: thêm key vào Colab Secrets, cấp quyền notebook.")
    previous_output = session.output
    try:
        if quiet: session.output = lambda event: None
        return session.execute("globals().pop(\"result\", None)\nfrom pjflim_agent.agent import MovieAgent\nfrom pjflim_agent.providers import Tmdb, ReviewDiscovery\nfrom pjflim_agent.omdb import Omdb\nfrom pjflim_agent.catalog import AgentError\n\nomdb_key = NOTEBOOK_SECRETS.pop(\"OMDB_API_KEY\", \"\")\ntmdb_token = NOTEBOOK_SECRETS.pop(\"TMDB_API_READ_TOKEN\", \"\")\nif \"_omdb_client\" not in globals() or not _omdb_client.matches_key(omdb_key):\n    _omdb_client = Omdb(api_key=omdb_key, mappings=OMDB_MAPPINGS)\nelse:\n    _omdb_client.mappings = OMDB_MAPPINGS\ntmdb = Tmdb(token=tmdb_token, mappings=TMDB_MAPPINGS)\ndel omdb_key, tmdb_token\ndiscovery = ReviewDiscovery(enabled=ENABLE_WEB_DISCOVERY)\nagent = MovieAgent(catalog, search, llm, tmdb=tmdb, discovery=discovery, omdb=_omdb_client)\ntry:\n    result = agent.ask(QUESTION, mode=MODE)\n    display(result[\"answer\"])\n    for card in result[\"cards\"]:\n        display({\"phim\": card[\"name\"], \"nam\": card[\"year\"], \"the_loai\": card[\"genres\"],\n                 \"nguon\": sorted({e[\"source\"][\"provider\"] for e in card[\"evidence\"]}),\n                 \"diem_ngoai\": card[\"external_ratings\"]})\n        display({\"lien_ket_review\": card.get(\"review_refs\", []), \"da_doc_bai\": False})\n    display({\"status\": result[\"status\"], \"demo\": result[\"demo\"], \"warnings\": result[\"warnings\"],\n             \"tool_calls\": result[\"tool_calls\"], \"live_checked\": False})\nexcept AgentError as error:\n    globals().pop(\"result\", None)\n    raise RuntimeError(error.code) from None\n\n\n__reply__ = result\n", {"QUESTION": question, "MODE": mode, "NOTEBOOK_SECRETS": secrets,
            "OMDB_MAPPINGS": OMDB_MAPPINGS, "TMDB_MAPPINGS": TMDB_MAPPINGS, "ENABLE_WEB_DISCOVERY": ENABLE_WEB_DISCOVERY})
    finally:
        session.output = previous_output
        secrets.clear()
if "CHAT_PANEL" in globals() and CHAT_PANEL is not None:
    CHAT_PANEL.close()
CHAT_PANEL = None
# Không giữ thành công cũ nếu lượt hỏi lại thất bại.
globals().pop("result", None)
result = ask_notebook(QUESTION, MODE, quiet=False)
if SHOW_CHAT_PANEL:
    from pjflim_agent.notebook_ui import build_chat_panel
    CHAT_PANEL = build_chat_panel(ask_notebook, MODE)
    if CHAT_PANEL is not None:
        display(CHAT_PANEL)
    else:
        print("Runtime chưa có ipywidgets. Sửa QUESTION rồi chạy lại cell8 để hỏi tiếp.")


## 9. Nghiệm thu

Chạy contract tests bằng FAISS và ChatOllama thật với HTTP/model fixtures; RUN_AGENT_EVALUATION=True thử thêm3câu demo bằng model/encoder hiện tại (có thể mất vài phút). Kết quả NEEDS_REVIEW không được ghi thành chất lượng đã đạt. Báo cáo chỉ lưu IDs/metrics/mã lỗi, không câu hỏi/hội thoại. Các nhóm live OMDb/TMDB/web chưa được đánh dấu đạt tự động. Dữ liệu nguồn chỉ là data; tool allowlist, schema, evidence registry và budget kiểm ngoài model.


In [ ]:
RUN_AGENT_EVALUATION = True  # 3 câu demo qua model hiện tại; False để chỉ chạy contract tests.
session.execute("import json\nimport unittest\nfrom datetime import datetime, timezone\nif \"result\" not in globals():\n    raise RuntimeError(\"RUN_CELL_8_SUCCESSFULLY_FIRST\")\nsuite = unittest.defaultTestLoader.discover(str(SOURCE / \"tests\"), pattern=\"test_*.py\")\ntests = unittest.TextTestRunner(verbosity=1).run(suite)\nif not tests.wasSuccessful():\n    raise RuntimeError(\"AGENT_CONTRACT_TESTS_FAILED\")\nfrom pjflim_agent.evaluation import evaluate_demo\nEVALUATION = evaluate_demo(agent, search) if RUN_AGENT_EVALUATION else {\"status\": \"SKIPPED\", \"reason\": \"DISABLED\"}\ndisplay({\"agent_evaluation\": EVALUATION})\nREPORT = {\"timestamp\": datetime.now(timezone.utc).isoformat(), \"hardware\": HARDWARE,\n          \"dependencies\": versions, \"model\": MODEL_MANIFEST, \"model_smoke\": MODEL_SMOKE, \"context_tokens\": CONTEXT_TOKENS,\n          \"embedding_model\": MODEL_CONFIG[\"embeddingModel\"], \"embedding_revision\": MODEL_CONFIG[\"embeddingRevision\"],\n          \"embedding_shape\": list(search.vectors.shape), \"contract_tests\": tests.testsRun,\n          \"agent_evaluation\": EVALUATION, \"report_schema_version\": 2,\n          \"contract_skipped\": len(tests.skipped), \"contract_failures\": len(tests.failures), \"contract_errors\": len(tests.errors),\n          \"demo\": catalog.snapshot.demo, \"agent_result_status\": result[\"status\"],\n          \"catalog_live_check\": False, \"omdb_live_test\": False, \"omdb_session_requests\": _omdb_client.session_calls,\n          \"tmdb_live_test\": False, \"web_live_test\": False}\n(WORKSPACE / \"runtime-report.json\").write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding=\"utf-8\")\ndisplay(REPORT)\nprint(\"Nếu dùng snapshot thật: luôn kiểm lại publication/quyền qua backend trước khi đưa kết quả lên website.\")\n", {"RUN_AGENT_EVALUATION": RUN_AGENT_EVALUATION})


## 10. Checkpoint và kết thúc

Tải checkpoint về PC hoặc tự lưu nơi được phép. Không có prompt/API key/model weights trong checkpoint; source metadata vẫn cần quyền lưu. Sau reset: bootstrap, nạp snapshot/checkpoint chưa hết hạn, kiểm lại model/index. Đặt STOP_OLLAMA=True để dừng đúng process notebook tạo. Không dùng notebook free làm API/RabbitMQ worker 24/7.


In [ ]:
DOWNLOAD_CHECKPOINT = False
DOWNLOAD_REPORT = False
STOP_OLLAMA = False
session.execute("from pjflim_agent.checkpoint import save_checkpoint\nfrom pjflim_agent.runtime import stop_ollama\narchive_path = save_checkpoint(WORKSPACE)\nprint(\"Checkpoint:\", archive_path, \"— không có key/prompt/model weights.\")\nif STOP_OLLAMA:\n    display(stop_ollama(WORKSPACE))\nelse:\n    print(\"Chạy lại cell 8 để hỏi tiếp. Đặt STOP_OLLAMA=True ở cell này khi kết thúc.\")\n", {"STOP_OLLAMA": STOP_OLLAMA})
if DOWNLOAD_CHECKPOINT:
    from google.colab import files
    files.download(str(WORKSPACE / "pj-flim-agent-checkpoint.zip"))
if DOWNLOAD_REPORT:
    from google.colab import files
    files.download(str(WORKSPACE / "runtime-report.json"))
if STOP_OLLAMA:
    session.close()
    if globals().get("CHAT_PANEL") is not None: CHAT_PANEL.close()
